# moe-engine v0.3.2 — T4 GPU Validation Notebook

This notebook runs the full moe-engine v0.3.2 validation suite on a real **T4 GPU**
(Google Colab or equivalent). It covers:

1. **Environment check** — confirm T4 is attached and CUDA is available  
2. **Install** — clone the refactored repo and install dependencies  
3. **Codebase verification** — confirm v0.3.2 fixes are present (Triton `K: tl.constexpr`, split modules, Pydantic config)  
4. **Smoke test** — `train.py --smoke` on the real Triton GPU path  
5. **Config validation** — test Pydantic `MoEConfig` system  
6. **Full test suite** — all 235 CPU tests + GPU-specific tests  
7. **GPU benchmark sweep** — real Triton kernel numbers at production-like shapes  
8. **CPU vs GPU comparison** — side-by-side throughput table  
9. **Throughput chart** — reproduce `router_throughput_gpu_v0_3_2.png`  
10. **Chaos pass-rate** — Scenario A (node kill) and Scenario B (storage stall)  
11. **Direct Triton sanity check** — H=4096, E=64, K=2 at production scale  
12. **Artifact packaging** — zip all outputs for download  

**Before running:** `Runtime → Change runtime type → T4 GPU`  

> **Note:** This notebook targets the v0.3.2 refactored codebase in  
> `moe-engine/notebooks/` (the zip you downloaded). It uses the zip as the  
> source of truth — NOT the GitHub repo, which may be stale.


## 0. Confirm T4 GPU is attached

Expect: `Tesla T4` with 15,360 MiB. If you see `No CUDA device`, go to Runtime → Change runtime type → T4 GPU → Restart.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print()
print('torch.__version__        =', torch.__version__)
print('torch.cuda.is_available  =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('torch.cuda.device_name   =', torch.cuda.get_device_name(0))
    print('torch.version.cuda       =', torch.version.cuda)
else:
    raise RuntimeError(
        'No CUDA device. Go to Runtime > Change runtime type > T4 GPU, '
        'then Runtime > Restart session, then re-run from the top.'
    )


## 1. Upload and install the v0.3.2 refactored codebase

Upload `moe_engine_v032_refactored.zip` using the Files panel (folder icon → upload),  
then run this cell. The zip contains the full refactored codebase with split  
distributed modules, Pydantic config, and all tests.

> **Why not `git clone`?** The GitHub repo may not yet reflect the v0.3.2  
> refactoring. The zip you downloaded is the authoritative source of truth.


In [ ]:
import os, zipfile, pathlib

ZIP_PATH = '/content/moe_engine_v032_refactored.zip'

# If not uploaded yet, try to find it
if not pathlib.Path(ZIP_PATH).exists():
    # Try common download locations
    for candidate in ['/content/*.zip', '/root/Downloads/*.zip']:
        import glob
        found = glob.glob(candidate)
        if found:
            ZIP_PATH = found[0]
            print(f'Found zip at: {ZIP_PATH}')
            break
    else:
        raise FileNotFoundError(
            'Upload moe_engine_v032_refactored.zip to /content/ first.\n'
            'Use the Files panel (folder icon) → Upload.'
        )

# Extract
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content')

# Find the moe-engine root inside the extracted tree
for candidate in [
    '/content/moe_upgraded/moe-engine',
    '/content/moe-engine',
]:
    if pathlib.Path(candidate).exists():
        MOE_ROOT = candidate
        break
else:
    raise RuntimeError('Could not find moe-engine root in extracted zip.')

print(f'moe-engine root: {MOE_ROOT}')
os.chdir(MOE_ROOT)
print(f'Working directory: {os.getcwd()}')


In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q pydantic typer  # for new Pydantic config + CLI
import sys; sys.path.insert(0, '.')
print('Installation complete.')


## 2. Verify v0.3.2 refactoring is present

This cell inspects the files on disk before running anything. It confirms:

- Triton kernels declare `K: tl.constexpr` (v0.3.2 bug fix)  
- `parallel_mesh.py` is the backward-compat shim (< 100 lines), not the monolith  
- New split modules exist: `mesh.py`, `tensor_parallel.py`, `expert_parallel.py`, etc.  
- Pydantic `MoEConfig` is present in `pkg/utils/config.py`  
- `pkg/models/moe.py` exists (model extracted from `train.py`)  


In [ ]:
import pathlib

checks = []

# Check 1: Triton K constexpr fix
router_src = pathlib.Path('pkg/kernels/moe_router.py').read_text()
k_count = router_src.count('K: tl.constexpr,')
checks.append(('Triton K: tl.constexpr in both kernels', k_count >= 2,
               f'found {k_count}, expected >= 2'))

# Check 2: parallel_mesh.py is the shim (< 100 lines)
pm_lines = len(pathlib.Path('pkg/distributed/parallel_mesh.py').read_text().splitlines())
checks.append(('parallel_mesh.py is shim (< 100 lines)', pm_lines < 100,
               f'has {pm_lines} lines'))

# Check 3: New split modules exist
for mod in ['mesh', 'tensor_parallel', 'expert_parallel',
             'pipeline_parallel', 'data_parallel', 'moe_layer']:
    p = pathlib.Path(f'pkg/distributed/{mod}.py')
    checks.append((f'pkg/distributed/{mod}.py exists', p.exists(), str(p)))

# Check 4: Pydantic MoEConfig
cfg_src = pathlib.Path('pkg/utils/config.py').read_text()
checks.append(('MoEConfig Pydantic class present', 'class MoEConfig' in cfg_src, ''))
checks.append(('ConfigValidationError present', 'class ConfigValidationError' in cfg_src, ''))

# Check 5: pkg/models/moe.py
checks.append(('pkg/models/moe.py exists',
               pathlib.Path('pkg/models/moe.py').exists(), ''))

# Check 6: test_config.py
checks.append(('tests/test_config.py exists',
               pathlib.Path('tests/test_config.py').exists(), ''))

# Check 7: scripts/cli.py
checks.append(('scripts/cli.py exists',
               pathlib.Path('scripts/cli.py').exists(), ''))

print(f'\nv0.3.2 refactoring verification ({len(checks)} checks):')
print('=' * 60)
passed = 0
for desc, ok, note in checks:
    status = '\033[92mPASS\033[0m' if ok else '\033[91mFAIL\033[0m'
    suffix = f'  ({note})' if note else ''
    print(f'  [{status}] {desc}{suffix}')
    if ok: passed += 1
print(f'\n{passed}/{len(checks)} checks passed.')
if passed < len(checks):
    raise RuntimeError('Some checks failed — verify the zip is the v0.3.2 refactored version.')
else:
    print('All checks passed. Proceeding with validation.')


## 3. Smoke test — Triton GPU path end-to-end

`train.py --smoke` with `configs/smoke.yaml` exercises:
- `MoEConfig.from_yaml()` (Pydantic validation)
- `build_topology()` (single-rank degenerate mesh)
- `ToyMoEModel` → `DistributedMoELayer` → `MoERouter` (Triton kernel on CUDA)
- `ElasticTrainerHarness` checkpoint
- `StructuredLogger` JSONL output

If this cell completes without `CompilationError` or `RuntimeError`, the v0.3.2  
Triton fix is validated end-to-end on a real GPU.


In [ ]:
!python train.py --config configs/smoke.yaml --smoke
print()
print('=' * 70)
print('step.jsonl (last 3 lines):')
print('=' * 70)
import subprocess, json
result = subprocess.run(['cat', '/tmp/moe-engine/logs/step.jsonl'],
                        capture_output=True, text=True)
for line in result.stdout.strip().split('\n')[-3:]:
    try:
        rec = json.loads(line)
        print(json.dumps(rec, indent=2))
    except Exception:
        print(line)


## 4. Config validation — Pydantic MoEConfig

Test the new Pydantic config system directly. Confirms:
- `smoke.yaml` and `default.yaml` load and validate
- Bad configs are rejected with field-level error messages
- Environment variable overrides work
- Legacy `load_config()` shim is backward-compatible


In [ ]:
from pkg.utils.config import MoEConfig, ConfigValidationError, load_config
import os

tests_passed = []

# Test 1: smoke.yaml
cfg = MoEConfig.from_yaml('configs/smoke.yaml')
assert cfg.model.hidden_dim == 32, f'Expected 32, got {cfg.model.hidden_dim}'
assert cfg.parallelism.world_size == 1
tests_passed.append('smoke.yaml loads and validates')

# Test 2: default.yaml
cfg2 = MoEConfig.from_yaml('configs/default.yaml')
assert cfg2.model.hidden_dim == 4096
assert cfg2.model.num_experts == 64
tests_passed.append('default.yaml loads (H=4096, E=64)')

# Test 3: top_k > num_experts rejected
try:
    MoEConfig.from_dict({'model': {'hidden_dim': 64, 'num_experts': 4, 'top_k': 8,
                                   'capacity_factor': 1.25, 'ffn_dim': 64,
                                   'vocab_size': 128, 'sequence_length': 8,
                                   'dtype': 'float32', 'num_layers': 2}})
    tests_passed.append('FAIL: should have caught top_k > num_experts')
except ConfigValidationError:
    tests_passed.append('top_k > num_experts correctly rejected')

# Test 4: env override
os.environ['MOE_MODEL__HIDDEN_DIM'] = '128'
cfg3 = MoEConfig.from_yaml('configs/smoke.yaml')
assert cfg3.model.hidden_dim == 128, f'Env override failed: got {cfg3.model.hidden_dim}'
del os.environ['MOE_MODEL__HIDDEN_DIM']
tests_passed.append('Environment variable override MOE_MODEL__HIDDEN_DIM=128')

# Test 5: legacy shim
legacy = load_config('configs/smoke.yaml')
assert legacy.raw['model']['hidden_dim'] == 32
assert legacy.typed().model.hidden_dim == 32
tests_passed.append('Legacy load_config() shim backward-compatible')

print(f'Config tests: {len(tests_passed)} passed')
for t in tests_passed:
    print(f'  [OK] {t}')


## 5. Full test suite — 235 CPU tests + GPU tests

Runs all tests marked `cpu` (no multi-process, no chaos).  
Expected: **235 passed, 1 skipped** (Triton GPU path test already runs on this T4),  
1 stochastic flake at seed=2 (pre-existing, unrelated to this refactoring).

Then runs the GPU-specific Triton constexpr tests separately.


In [ ]:
!pytest tests/ \
  --ignore=tests/test_chaos.py \
  --ignore=tests/test_smoke_e2e.py \
  -m cpu \
  -k 'not (2rank or multiprocess or distributed_invariants)' \
  -q --tb=short 2>&1 | tee /content/test_suite_cpu_output.txt | tail -20


In [ ]:
# GPU-specific: Triton constexpr regression tests (require Triton + CUDA)
!pytest tests/test_kernels.py -v -k 'constexpr or gpu' 2>&1 | tail -15


## 6. GPU benchmark sweep — real Triton kernel throughput

Runs `benchmarks/run_benchmark.py --cuda` to measure the Triton router kernel  
on the T4. This generates the numbers in `RESULTS.md` and `BENCHMARKS.md`.

The `--cuda` flag switches from the fp64 reference path to the real  
`moe_topk_route_fwd` / `moe_topk_route_bwd` Triton kernels.


In [ ]:
# Apply device-mismatch patch (needed when CPU and CUDA tensors mix in reference path)
import pathlib, re

router_path = pathlib.Path('pkg/kernels/moe_router.py')
src = router_path.read_text()

# Patch 1: FP64 reference path
src = src.replace(
    'logits = t64 @ g64',
    'logits = t64 @ g64.to(tokens.device)'
)
# Patch 2: FP32 reference path
src = src.replace(
    'logits_fp32 = (flat.float() @ gate_w.float())',
    'logits_fp32 = (flat.float() @ gate_w.float().to(tokens.device))'
)

router_path.write_text(src)
print('Device-mismatch patches applied.')


In [ ]:
!python benchmarks/run_benchmark.py \
    --cuda \
    --json /content/gpu_results.json \
    --csv  /content/gpu_results.csv
print('\nGPU benchmark complete.')


## 7. CPU benchmark sweep (for CPU vs GPU comparison)

In [ ]:
!python benchmarks/run_benchmark.py \
    --json /content/cpu_results.json \
    --csv  /content/cpu_results.csv
print('\nCPU benchmark complete.')


## 8. CPU vs GPU throughput comparison

Side-by-side table comparing the fp64 CPU reference path versus the  
Triton GPU kernel, for all `router_fwd` configurations.  

The headline result is **~80× GPU speedup** at N=4096, H=2048, E=64, K=4 —  
the most production-relevant configuration.


In [ ]:
import json

cpu_data = json.load(open('/content/cpu_results.json'))
gpu_data = json.load(open('/content/gpu_results.json'))

cpu_map = {(d['name'], d['N'], d['H'], d['E'], d['K']): d
           for d in cpu_data if d['name'].startswith('router')}
gpu_map = {(d['name'], d['N'], d['H'], d['E'], d['K']): d
           for d in gpu_data if d['name'].startswith('router')}

print(f"{'Config':<38} {'CPU (M tok/s)':>14} {'GPU (M tok/s)':>14} {'Speedup':>10}")
print('-' * 80)

for key in sorted(set(cpu_map) & set(gpu_map)):
    name, n, h, e, k = key
    c = cpu_map[key]['tokens_per_sec']
    g = gpu_map[key]['tokens_per_sec']
    tag = f'{name} N={n},H={h},E={e},K={k}'
    speedup = g / c
    highlight = ' ◄── 80x!' if speedup > 70 else ''
    print(f'  {tag:<36} {c/1e6:>13.3f} {g/1e6:>13.3f} {speedup:>9.1f}x{highlight}')


## 9. Generate throughput chart (v0.3.2)

Reproduces `benchmarks/charts/router_throughput_gpu_v0_3_2.png`  
from the real GPU data, showing both forward-only and forward+backward  
throughput on a log scale across all benchmark configurations.


In [ ]:
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

gpu = json.load(open('/content/gpu_results.json'))
cpu = json.load(open('/content/cpu_results.json'))

router_fwd_gpu  = [d for d in gpu if d['name'] == 'router_fwd'  and d['device'] == 'cuda']
router_bwd_gpu  = [d for d in gpu if d['name'] == 'router_fwd_bwd' and d['device'] == 'cuda']
router_fwd_cpu  = [d for d in cpu if d['name'] == 'router_fwd'  and d['device'] == 'cpu']

configs  = [(d['N'], d['H'], d['E'], d['K']) for d in router_fwd_gpu]
labels   = [f'N={n}\nH={h},E={e},K={k}' for n, h, e, k in configs]
x        = list(range(len(configs)))

fwd_gpu  = [d['tokens_per_sec'] / 1e6 for d in router_fwd_gpu]
bwd_gpu  = [d['tokens_per_sec'] / 1e6 for d in router_bwd_gpu]
fwd_cpu  = [d['tokens_per_sec'] / 1e6 for d in router_fwd_cpu]

fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(x, fwd_gpu, marker='o', linewidth=2.2, color='#1f77b4',
        label='router_fwd (Triton, GPU)')
ax.plot(x, bwd_gpu, marker='s', linewidth=2.2, color='#ff7f0e',
        label='router_fwd_bwd (Triton, GPU)')
ax.plot(x, fwd_cpu, marker='^', linewidth=1.5, color='#2ca02c',
        linestyle='--', label='router_fwd (fp64 ref, CPU)')

# Annotate max speedup
max_speedup_idx = int(np.argmax([g/c for g, c in zip(fwd_gpu, fwd_cpu)]))
speedup = fwd_gpu[max_speedup_idx] / fwd_cpu[max_speedup_idx]
ax.annotate(f'{speedup:.0f}× vs CPU',
            xy=(max_speedup_idx, fwd_gpu[max_speedup_idx]),
            xytext=(max_speedup_idx - 0.4, fwd_gpu[max_speedup_idx] * 1.5),
            fontsize=10, color='#1f77b4',
            arrowprops=dict(arrowstyle='->', color='#1f77b4'))

ax.set_yscale('log')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylabel('Throughput (M tokens/sec, log scale)', fontsize=11)
ax.set_title(
    'moe-engine Router Throughput — T4 GPU (Triton) vs CPU Reference\n'
    '(v0.3.2, real measurements, June 2026)',
    fontsize=12
)
ax.legend(fontsize=9)
ax.grid(True, which='both', alpha=0.3)

out_path = '/content/router_throughput_gpu_v0_3_2.png'
plt.tight_layout()
plt.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Chart saved to {out_path}')

# Display inline
from IPython.display import Image
Image(out_path)


## 10. Chaos pass-rate measurement

Tests fault-tolerance under two failure scenarios:

**Scenario B (storage stall):** A 10-second injected I/O stall during async  
checkpoint commit. Expected: **10/10 passes (100%)**. This is stable.

**Scenario A (node kill + recovery):** SIGKILL sent to one rank; TorchElastic  
restarts it; expert resharding. Expected: **~85% pass rate** due to a known  
Gloo `connectFullMesh` race in containerised environments. Non-blocking in CI.  
Planned fix: replace Gloo with NCCL in chaos harness (v0.4).


In [ ]:
# Scenario B: storage stall (should be 10/10)
!GLOO_SOCKET_IFNAME=lo pytest tests/test_chaos.py \
    -v -m chaos -k 'scenario_b' --count=10 \
    2>&1 | tee /content/chaos_scenario_b_output.txt | tail -20


In [ ]:
# Scenario A: node kill + recovery (expect ~85%)
!CHAOS_FAULT_TOLERANT=1 GLOO_SOCKET_IFNAME=lo \
    pytest tests/test_chaos.py \
    -v -m chaos -k 'scenario_a' --count=20 \
    2>&1 | tee /content/chaos_scenario_a_output.txt | tail -20


In [ ]:
import re

def count_pass_fail(path):
    try:
        text = open(path).read()
        passed = len(re.findall(r'PASSED', text))
        failed = len(re.findall(r'FAILED', text))
        return passed, failed
    except FileNotFoundError:
        return 0, 0

a_pass, a_fail = count_pass_fail('/content/chaos_scenario_a_output.txt')
b_pass, b_fail = count_pass_fail('/content/chaos_scenario_b_output.txt')

total_a = a_pass + a_fail
total_b = b_pass + b_fail

print('Chaos pass-rate summary:')
print(f'  Scenario A (node kill):    {a_pass}/{total_a} passed  '
      f'({100*a_pass/max(total_a,1):.0f}%)  — expect ~85%')
print(f'  Scenario B (storage stall): {b_pass}/{total_b} passed  '
      f'({100*b_pass/max(total_b,1):.0f}%)  — expect 100%')

if total_b > 0 and b_pass < total_b:
    print('\n⚠️  Scenario B regression — expected 10/10.')
elif total_b > 0:
    print('\n✅  Scenario B: 100% pass rate confirmed.')


## 11. Direct Triton sanity check at production scale

Calls `MoERouter` directly at **H=4096, E=64, K=2** — the `default.yaml`  
production configuration. Confirms:
- `used_triton = True` (real Triton kernel, not the fp64 fallback)
- `dispatch_cnt.sum() == N × K` (token conservation)
- No NaN in indices or weights
- `idx ∈ [0, E)` (index validity)


In [ ]:
import torch, sys
sys.path.insert(0, '.')
from pkg.kernels.moe_router import MoERouter

torch.manual_seed(42)
device = 'cuda'
N, H, E, K = 2048, 4096, 64, 2

router = MoERouter(hidden_dim=H, num_experts=E, top_k=K).to(device)
tokens = torch.randn(N, H, device=device)

with torch.no_grad():
    idx, w, dispatch_cnt = router(tokens)

# Assertions
assert idx.shape == (N, K),            f'idx shape: {idx.shape}'
assert w.shape   == (N, K),            f'w shape: {w.shape}'
assert int(dispatch_cnt.sum()) == N*K, f'conservation: {int(dispatch_cnt.sum())} != {N*K}'
assert not torch.isnan(w).any(),       'NaN in combine weights'
assert (idx >= 0).all() and (idx < E).all(), 'idx out of bounds'
assert torch.allclose(w.sum(-1), torch.ones(N, device=device), atol=1e-4), \
    'weights do not sum to 1'

print('Production-scale Triton sanity check:')
print(f'  idx.shape              : {idx.shape}')
print(f'  w.shape                : {w.shape}')
print(f'  dispatch_cnt.sum()     : {int(dispatch_cnt.sum())}  (expected {N*K})')
print(f'  used_triton            : {router.last_profile.used_triton}')
print(f'  kernel_ms              : {router.last_profile.kernel_ms:.4f}')
print(f'  expert_load_imbalance  : {router.last_profile.expert_load_imbalance:.4f}')
print(f'  router_z_loss          : {router.last_profile.router_z_loss:.4f}')
print()
print('PASS: All invariants hold at H=4096, E=64, K=2 on T4.')


## 12. Pydantic config CLI test

Test `scripts/cli.py validate` and `scripts/cli.py info` commands.


In [ ]:
!python scripts/cli.py validate configs/
print()
!python scripts/cli.py info


## 13. Package all artifacts for download

Run this before closing the tab — Colab's filesystem is ephemeral.  
Everything is packaged into `moe_engine_t4_validation_v032.zip`.


In [ ]:
import shutil, os, pathlib

artifact_dir = pathlib.Path('/content/moe_engine_t4_artifacts')
artifact_dir.mkdir(exist_ok=True)

files_to_collect = [
    ('/content/gpu_results.json',              'gpu_results.json'),
    ('/content/gpu_results.csv',               'gpu_results.csv'),
    ('/content/cpu_results.json',              'cpu_results.json'),
    ('/content/cpu_results.csv',               'cpu_results.csv'),
    ('/content/router_throughput_gpu_v0_3_2.png', 'router_throughput_gpu_v0_3_2.png'),
    ('/content/test_suite_cpu_output.txt',     'test_suite_cpu_output.txt'),
    ('/content/chaos_scenario_a_output.txt',   'chaos_scenario_a_output.txt'),
    ('/content/chaos_scenario_b_output.txt',   'chaos_scenario_b_output.txt'),
    ('/tmp/moe-engine/logs/step.jsonl',        'smoke_step.jsonl'),
]

collected = []
for src, dst in files_to_collect:
    try:
        shutil.copy(src, artifact_dir / dst)
        collected.append(dst)
    except FileNotFoundError:
        print(f'  [skip] {dst} not found (run the relevant cell first)')

zip_path = '/content/moe_engine_t4_validation_v032'
shutil.make_archive(zip_path, 'zip', artifact_dir)
print(f'Collected: {collected}')
print(f'Zipped to: {zip_path}.zip')


In [ ]:
from google.colab import files
files.download('/content/moe_engine_t4_validation_v032.zip')


---

## What to do with the downloaded zip

The `moe_engine_t4_validation_v032.zip` contains:

| File | Purpose |
|------|---------|
| `gpu_results.json` | Raw T4 GPU benchmark data — all real numbers |
| `cpu_results.json` | CPU reference path data for comparison |
| `router_throughput_gpu_v0_3_2.png` | Throughput chart (place in `benchmarks/charts/`) |
| `test_suite_cpu_output.txt` | Full pytest output (235 passed) |
| `chaos_scenario_a_output.txt` | Scenario A raw output |
| `chaos_scenario_b_output.txt` | Scenario B raw output (10/10) |
| `smoke_step.jsonl` | JSONL telemetry from smoke train run |

All numbers in `RESULTS.md` and `benchmarks/BENCHMARKS.md` are derived  
from `gpu_results.json` and `cpu_results.json` in this zip.
